# Agent Memory Toolkit – Function App (Remote Processor) Demo

This notebook demonstrates the **`DurableFunctionProcessor`** hand-off pattern: the SDK only
writes raw turns to Cosmos DB, and a sibling **Azure Function App** (deployed from the
`function_app/` folder via `azd up`) picks them up from the Cosmos change feed and produces
summaries, facts, episodic memories, and procedural rules asynchronously — server-side.

## When to use this pattern

| Use case | Recommended processor |
|---|---|
| Local dev, scripts, single agent process | **`InProcessProcessor`** (default) — no extra infra |
| Production, multi-agent fleets, server-managed processing, audit trail | **`DurableFunctionProcessor`** — durable, scalable, isolated |

## Prerequisites

1. **Function App deployed** — run `azd up` (or deploy `function_app/` manually) against the same Cosmos
   account/database/container the SDK targets.
2. **Function App env vars** include `THREAD_SUMMARY_EVERY_N`, `FACT_EXTRACTION_EVERY_N`,
   `USER_SUMMARY_EVERY_N` (defaults: 4 / 6 / 10). Each one ≥ 1 enables the corresponding orchestrator.
3. **`.env`** in this repo has `COSMOS_DB_ENDPOINT`, `AI_FOUNDRY_ENDPOINT`, deployment names. The Function App
   uses managed identity to reach the same Cosmos + AI Foundry resources.

## What the SDK does in this mode

* `add_cosmos(..., memory_type="turn")` → writes the raw turn to Cosmos.
* The change-feed-triggered Function App reads new turns and runs orchestrators.
* `process_now()` is a **debug-logged no-op** — the Function App owns processing.
* `process_now_and_wait()` polls Cosmos for the summary doc; useful for demos / tests (RU-costly).

## 1. Setup

In [ ]:
import os, time, uuid
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

from agent_memory_toolkit import CosmosMemoryClient, DurableFunctionProcessor

print("Cosmos endpoint :", os.environ["COSMOS_DB_ENDPOINT"])
print("AI Foundry      :", os.environ["AI_FOUNDRY_ENDPOINT"])
print("Chat deployment :", os.environ["AI_FOUNDRY_CHAT_DEPLOYMENT_NAME"])
print("Embed deployment:", os.environ["AI_FOUNDRY_EMBEDDING_DEPLOYMENT_NAME"])

## 2. Construct the client with `DurableFunctionProcessor`

The single change vs. the in-process demo is `processor=DurableFunctionProcessor()`. After this, the
SDK **never invokes the LLM or embeddings client itself** — it only writes raw turns to Cosmos.

In [ ]:
memory = CosmosMemoryClient(
    cosmos_endpoint=os.environ["COSMOS_DB_ENDPOINT"],
    cosmos_key=os.environ.get("COSMOS_DB_KEY"),  # falls back to DefaultAzureCredential
    cosmos_database=os.environ.get("COSMOS_DB_DATABASE", "ai_memory"),
    cosmos_container=os.environ.get("COSMOS_DB_CONTAINER", "memories"),
    ai_foundry_endpoint=os.environ["AI_FOUNDRY_ENDPOINT"],
    chat_deployment_name=os.environ["AI_FOUNDRY_CHAT_DEPLOYMENT_NAME"],
    embedding_deployment_name=os.environ["AI_FOUNDRY_EMBEDDING_DEPLOYMENT_NAME"],
    # ── The hand-off ──
    processor=DurableFunctionProcessor(),
)
print(f"Processor: {type(memory._processor).__name__}")

## 3. Write a conversation that should trigger the orchestrators

We use a **unique `USER_ID`** so we do not inherit derived memories from prior runs.
A unique `THREAD_ID` ensures the change-feed orchestrator processes a clean batch.

In [ ]:
USER_ID   = f"fa-demo-{uuid.uuid4().hex[:8]}"
THREAD_ID = str(uuid.uuid4())
print(f"USER_ID   = {USER_ID}")
print(f"THREAD_ID = {THREAD_ID}")

transcript = [
    ("user",  "What's the weather like in Seattle this weekend?"),
    ("agent", "Around 55°F with partly cloudy skies on Saturday and light rain on Sunday."),
    ("user",  "Can you book me a weekend trip there? Flights under $300, hotel under $200."),
    ("agent", "I found a $275 Alaska Airlines round-trip and a $185/night hotel in Belltown."),
    ("user",  "Whenever you book a flight for me, always book an aisle seat — never window or middle."),
    ("agent", "Got it — aisle seats only."),
    ("user",  "For trip planning, my workflow is: first weather, then flights, then hotel."),
    ("agent", "Noted — weather → flights → hotel."),
    ("user",  "Never book me into anything between midnight and 6am unless I explicitly approve."),
    ("agent", "Understood — no overnight bookings without your approval."),
]
for role, content in transcript:
    memory.add_cosmos(
        user_id=USER_ID,
        thread_id=THREAD_ID,
        role=role,
        content=content,
        memory_type="turn",
    )
    print(f"  wrote {role:>5}: {content[:60]}{'…' if len(content) > 60 else ''}")

## 4. Verify the SDK did NOT process locally

`process_now()` is a debug-logged no-op when using `DurableFunctionProcessor`. The Function App's change-feed
trigger fires asynchronously — usually within a second or two.

In [ ]:
# No-op: the function app owns processing.
memory.process_now(user_id=USER_ID, thread_id=THREAD_ID)
print("process_now() returned — no LLM call was made by the SDK.")

## 5. Wait for the Function App to produce a summary

`process_now_and_wait` polls Cosmos for the thread's summary doc until `timeout` seconds. This is **RU-costly**
(repeated `get_memories(memory_types=['summary'])` queries) — only use it for demos and tests.

> If this returns `False`, check that:
> * The Function App is deployed and running.
> * `THREAD_SUMMARY_EVERY_N` is `≥ 1` and `≤` the number of turns written above.
> * The Function App's managed identity has Cosmos data + Cognitive Services OpenAI User roles.

In [ ]:
print("Polling Cosmos for the auto-generated summary…")
ok = memory.process_now_and_wait(user_id=USER_ID, thread_id=THREAD_ID, timeout=120.0)
print(f"Summary available: {ok}")

## 6. Inspect what the Function App produced

After the orchestrators run, the Function App writes:

| Type | When | Rule of thumb |
|---|---|---|
| `summary` | Every `THREAD_SUMMARY_EVERY_N` turns per thread | Compressed conversation state |
| `fact` | Every `FACT_EXTRACTION_EVERY_N` turns | Stable facts about the user |
| `episodic` | Same orchestrator as facts | Past events |
| `procedural` | Same orchestrator as facts | Behavioral rules / workflows |
| `user_summary` | Every `USER_SUMMARY_EVERY_N` thread summaries | Cross-thread profile |

In [ ]:
# All derived memories for this user, written by the function app.
# Fact / episodic / procedural extraction runs in a separate orchestrator that
# lags the summary by a few seconds. Poll briefly for them to land.
import time

def _show():
    counts = {}
    for mt in ("summary", "fact", "episodic", "procedural", "user_summary"):
        docs = memory.get_memories(user_id=USER_ID, memory_types=[mt])
        counts[mt] = len(docs)
        print(f"\n{mt.upper()}S ({len(docs)}):")
        for d in docs:
            print(f"  • [{d['id'][:32]}…] {d['content'][:90]}")
    return counts

print("Initial state:")
counts = _show()

# If facts haven't landed yet, give the extraction orchestrator a bit longer.
for attempt in range(6):
    if counts.get("fact", 0) > 0 or counts.get("procedural", 0) > 0:
        break
    print(f"\n... fact / procedural extraction still in flight (attempt {attempt+1}/6, sleeping 15s)")
    time.sleep(15)
    counts = {mt: len(memory.get_memories(user_id=USER_ID, memory_types=[mt]))
              for mt in ("summary", "fact", "episodic", "procedural", "user_summary")}

print("\n=== Final state ===")
_show()


## 7. Going further

* **Per-turn embedding**: the SDK auto-embeds non-`turn` documents you `add_cosmos` directly. Raw `turn`
  records are intentionally not embedded — the function app does that during summary/extraction.
* **Search across function-app-produced memories**: works exactly as in the in-process demo:
  `memory.search_cosmos(search_terms="…", memory_types=["fact"], user_id=USER_ID)`.
* **Switch back to in-process** for ad-hoc work: instantiate the client without the `processor=` kwarg
  (or pass `InProcessProcessor()` explicitly) and use `generate_thread_summary` / `extract_memories`
  directly.